In [ ]:
import numpy as np
import xarray as xr

def l_moments(data):
    """Compute L-moments (L1, L2, L3) for a given 1D array."""
    data = np.sort(data)
    n = len(data)
    if n < 3:
        return np.nan, np.nan, np.nan  # Not enough data for L3
    
    # Compute probability weighted moments
    b0 = np.mean(data)
    b1 = np.sum((np.arange(n) / (n - 1)) * data) / n
    b2 = np.sum(((np.arange(n) * (np.arange(n) - 1)) / ((n - 1) * (n - 2))) * data) / n
    
    # Compute L-moments
    L1 = b0
    L2 = 2 * b1 - b0
    L3 = 6 * b2 - 6 * b1 + b0
    
    return L1, L2, L3

def compute_l_skewness(da):
    """Compute L-skewness for an xarray DataArray along the time dimension."""
    def l_skewness_func(x):
        _, L2, L3 = l_moments(x)
        return L3 / L2 if L2 != 0 else np.nan
    
    return xr.apply_ufunc(l_skewness_func, da, input_core_dims=[['time']], vectorize=True)

# Example usage with synthetic precipitation data
lat, lon, time = 10, 10, 50
np.random.seed(42)
precip_data = np.random.gamma(shape=2, scale=2, size=(lat, lon, time))

da = xr.DataArray(precip_data, dims=["lat", "lon", "time"], 
                  coords={"lat": np.linspace(-90, 90, lat),
                          "lon": np.linspace(-180, 180, lon),
                          "time": np.arange(time)})

# Compute L-skewness
l_skewness = compute_l_skewness(da)
print(l_skewness)
